In [1]:
pip install langgraph langsmith langchain langchain_groq langchain_core langchain_openai langchain_community langchain-text-splitters bs4 IPython langchain_tavily

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\Abhineet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key = "", model_name = "llama-3.3-70b-versatile")
print(llm)

profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x00000231EC685940> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000231EC686660> model_name='llama-3.3-70b-versatile' model_kwargs={} groq_api_key=SecretStr('**********')


In [3]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    #create a variable messages of type annotated that set this is list and function call add_messages 
    #that add/append messages to the list on every add_messages the state is updated
    #usefull for 
    #Chatbots that need to remember conversation history
    #AI assistants that maintain context over multiple interactions
    messages:Annotated[list, add_messages]

In [4]:
# a tool that does multiply
def multiplyTool(a:int, b:int) -> int:
   """Multiply a and b

   Args:
        a(int) first int
        b(int) second int

    Returns
        int: output int   
   """
   return a * b

In [ ]:
from langchain_tavily import TavilySearch
import os

os.environ["TAVILY_API_KEY"] = ""
toolSearch = TavilySearch(max_results = 2)

#creating an array of such tools and binding with llm
tools = [toolSearch, multiplyTool]
llm_with_tool = llm.bind_tools(tools)

In [6]:
def chatbot(state:State):
    return {"messages": llm_with_tool.invoke(state["messages"])}
    #it is invoking the user query and returning the messages

In [7]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)
print(graph_builder)

In [8]:
from langgraph.prebuilt import ToolNode, tools_condition

In [9]:
graph_builder.add_node(chatbot, "chatbot")
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)

graph_builder.add_edge("tools", "chatbot")

In [10]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
while True:
  user_input = input("User: ")
  if user_input.lower() in ['quit', 'q']:
    print("Chat Ended")
    break
  events = graph.stream({"messages": ("user", user_input)}, stream_mode="values")
  for event in events:
    event["messages"][-1].pretty_print()